# FASE 3: Pemodelan Baseline (Random Forest & XGBoost) + Hyperparameter Tuning
## Prediksi PM2.5 di Cekungan Bandung
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini melatih model *Machine Learning* klasik berbasis *Ensemble Trees*, yaitu **Random Forest** dan **XGBoost**, sebagai model baseline (tolok ukur). Kedua model ini dioptimasi menggunakan **Hyperparameter Tuning** agar perbandingan dengan LSTM/BiLSTM bersifat adil (*fair comparison*).

**Alur Notebook:**
1. Load data Train, Validation, dan Test yang sudah diproses.
2. Melatih Random Forest dengan parameter default sebagai *baseline awal*.
3. Melakukan **Hyperparameter Tuning** menggunakan data Validasi.
4. Mengevaluasi model terbaik pada data Testing.
5. Menyimpan model yang sudah dioptimasi.


## 1. Import Library


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
import time

# Evaluasi
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Model
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print("[OK] Library loaded.")


## 2. Load Data Processed & Scaler

Memuat tiga partisi data (Train/Validation/Test) yang sudah bersih dan terskala dari hasil Notebook 02.


In [ ]:
DATA_DIR = '/content/drive/MyDrive/KP_BRIN/data/processed'

# Load Dataset (3 partisi)
train_scaled = pd.read_csv(os.path.join(DATA_DIR, 'train_scaled.csv'), index_col=0, parse_dates=True)
val_scaled   = pd.read_csv(os.path.join(DATA_DIR, 'val_scaled.csv'), index_col=0, parse_dates=True)
test_scaled  = pd.read_csv(os.path.join(DATA_DIR, 'test_scaled.csv'), index_col=0, parse_dates=True)

# Load Scaler Target (Y) untuk inversi nanti
scaler_y = joblib.load(os.path.join(DATA_DIR, 'scaler_y.pkl'))

# Pisahkan Fitur (X) dan Target (y)
target_col = 'pm25'

X_train = train_scaled.drop(columns=[target_col])
y_train = train_scaled[target_col]

X_val = val_scaled.drop(columns=[target_col])
y_val = val_scaled[target_col]

X_test = test_scaled.drop(columns=[target_col])
y_test = test_scaled[target_col]

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")


## 3. Fungsi Evaluasi (Inversi ke Satuan Asli)

Semua model memprediksi dalam skala [0, 1]. Agar metrik akurasi bermakna secara fisik, kita harus mengubah prediksi kembali ke satuan $\mu g/m^3$ menggunakan `inverse_transform` sebelum menghitung RMSE, MAE, dan R².


In [ ]:
def evaluate_model(model_name, y_true_scaled, y_pred_scaled, verbose=True):
    """Mengevaluasi model dengan menginversi skala kembali ke ug/m3."""
    y_true_inv = scaler_y.inverse_transform(y_true_scaled.values.reshape(-1, 1)).flatten()
    y_pred_inv = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
    mae  = mean_absolute_error(y_true_inv, y_pred_inv)
    r2   = r2_score(y_true_inv, y_pred_inv)

    if verbose:
        print(f"=== Kinerja Model: {model_name} ===")
        print(f"  RMSE : {rmse:.4f} ug/m3")
        print(f"  MAE  : {mae:.4f} ug/m3")
        print(f"  R2   : {r2:.4f}")

    return {'name': model_name, 'rmse': rmse, 'mae': mae, 'r2': r2,
            'y_true': y_true_inv, 'y_pred': y_pred_inv}


## 4. Random Forest: Default + Hyperparameter Tuning

### 4.1 Baseline (Parameter Default)
Langkah pertama: melatih Random Forest dengan konfigurasi standar untuk mendapatkan *baseline awal*.


In [ ]:
print("--- Random Forest: Parameter Default ---")
rf_default = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

start = time.time()
rf_default.fit(X_train, y_train)
print(f"Training selesai ({time.time()-start:.1f} detik)")

# Evaluasi pada Validation Set
y_pred_rf_val = rf_default.predict(X_val)
rf_default_val_result = evaluate_model("RF Default (Validasi)", y_val, y_pred_rf_val)

# Evaluasi pada Test Set
y_pred_rf_test = rf_default.predict(X_test)
rf_default_test_result = evaluate_model("RF Default (Testing)", y_test, y_pred_rf_test)


### 4.2 Hyperparameter Tuning Random Forest

Kita mencoba beberapa kombinasi hyperparameter dan memilih yang memberikan **RMSE terendah pada data Validasi**. Tuning dilakukan secara manual (*Grid Search*) pada parameter berikut:

- `n_estimators`: Jumlah pohon keputusan [100, 200, 300]
- `max_depth`: Kedalaman maksimum pohon [10, 20, None]
- `min_samples_split`: Jumlah sampel minimum untuk membelah node [2, 5, 10]


In [ ]:
print("--- Hyperparameter Tuning Random Forest ---")
print("Mengevaluasi kombinasi parameter pada data Validasi...\n")

rf_param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

best_rf_rmse = float('inf')
best_rf_params = None
best_rf_model = None
rf_results = []

total_combos = (len(rf_param_grid['n_estimators']) *
                len(rf_param_grid['max_depth']) *
                len(rf_param_grid['min_samples_split']))
combo_idx = 0

for n_est in rf_param_grid['n_estimators']:
    for max_d in rf_param_grid['max_depth']:
        for min_split in rf_param_grid['min_samples_split']:
            combo_idx += 1
            model = RandomForestRegressor(
                n_estimators=n_est, max_depth=max_d,
                min_samples_split=min_split,
                random_state=42, n_jobs=-1
            )
            model.fit(X_train, y_train)
            y_pred_val = model.predict(X_val)
            res = evaluate_model(f"RF [{combo_idx}/{total_combos}]", y_val, y_pred_val, verbose=False)

            rf_results.append({
                'n_estimators': n_est,
                'max_depth': max_d,
                'min_samples_split': min_split,
                'val_rmse': res['rmse'],
                'val_mae': res['mae'],
                'val_r2': res['r2']
            })

            if res['rmse'] < best_rf_rmse:
                best_rf_rmse = res['rmse']
                best_rf_params = {'n_estimators': n_est, 'max_depth': max_d, 'min_samples_split': min_split}
                best_rf_model = model

            print(f"  [{combo_idx:2d}/{total_combos}] n_est={n_est:3d} | max_d={str(max_d):>5s} | min_split={min_split:2d} --> Val RMSE={res['rmse']:.4f}")

print(f"\n>> Best RF Params: {best_rf_params}")
print(f">> Best RF Val RMSE: {best_rf_rmse:.4f}")


In [ ]:
# Evaluasi model RF terbaik pada Test Set
print("--- Evaluasi RF Terbaik pada Data Testing ---")
y_pred_rf_best_test = best_rf_model.predict(X_test)
rf_tuned_test_result = evaluate_model("RF Tuned (Testing)", y_test, y_pred_rf_best_test)

print(f"\nPerbandingan RF Default vs RF Tuned (pada Test Set):")
print(f"  Default --> RMSE: {rf_default_test_result['rmse']:.4f} | MAE: {rf_default_test_result['mae']:.4f} | R2: {rf_default_test_result['r2']:.4f}")
print(f"  Tuned   --> RMSE: {rf_tuned_test_result['rmse']:.4f} | MAE: {rf_tuned_test_result['mae']:.4f} | R2: {rf_tuned_test_result['r2']:.4f}")


## 5. XGBoost: Default + Hyperparameter Tuning

### 5.1 Baseline (Parameter Default)


In [ ]:
print("--- XGBoost: Parameter Default ---")
xgb_default = xgb.XGBRegressor(
    n_estimators=100, learning_rate=0.1,
    max_depth=6, random_state=42, n_jobs=-1
)

start = time.time()
xgb_default.fit(X_train, y_train)
print(f"Training selesai ({time.time()-start:.1f} detik)")

# Evaluasi Validation
y_pred_xgb_val = xgb_default.predict(X_val)
xgb_default_val_result = evaluate_model("XGB Default (Validasi)", y_val, y_pred_xgb_val)

# Evaluasi Testing
y_pred_xgb_test = xgb_default.predict(X_test)
xgb_default_test_result = evaluate_model("XGB Default (Testing)", y_test, y_pred_xgb_test)


### 5.2 Hyperparameter Tuning XGBoost

Parameter yang dioptimasi:
- `n_estimators`: Jumlah iterasi boosting [100, 200, 300]
- `learning_rate`: Kecepatan pembelajaran [0.01, 0.05, 0.1]
- `max_depth`: Kedalaman pohon [4, 6, 8]


In [ ]:
print("--- Hyperparameter Tuning XGBoost ---")
print("Mengevaluasi kombinasi parameter pada data Validasi...\n")

xgb_param_grid = {
    'n_estimators':  [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth':     [4, 6, 8]
}

best_xgb_rmse = float('inf')
best_xgb_params = None
best_xgb_model = None
xgb_results = []

total_combos = (len(xgb_param_grid['n_estimators']) *
                len(xgb_param_grid['learning_rate']) *
                len(xgb_param_grid['max_depth']))
combo_idx = 0

for n_est in xgb_param_grid['n_estimators']:
    for lr in xgb_param_grid['learning_rate']:
        for max_d in xgb_param_grid['max_depth']:
            combo_idx += 1
            model = xgb.XGBRegressor(
                n_estimators=n_est, learning_rate=lr,
                max_depth=max_d, random_state=42, n_jobs=-1
            )
            model.fit(X_train, y_train)
            y_pred_val = model.predict(X_val)
            res = evaluate_model(f"XGB [{combo_idx}/{total_combos}]", y_val, y_pred_val, verbose=False)

            xgb_results.append({
                'n_estimators': n_est,
                'learning_rate': lr,
                'max_depth': max_d,
                'val_rmse': res['rmse'],
                'val_mae': res['mae'],
                'val_r2': res['r2']
            })

            if res['rmse'] < best_xgb_rmse:
                best_xgb_rmse = res['rmse']
                best_xgb_params = {'n_estimators': n_est, 'learning_rate': lr, 'max_depth': max_d}
                best_xgb_model = model

            print(f"  [{combo_idx:2d}/{total_combos}] n_est={n_est:3d} | lr={lr:.2f} | max_d={max_d} --> Val RMSE={res['rmse']:.4f}")

print(f"\n>> Best XGB Params: {best_xgb_params}")
print(f">> Best XGB Val RMSE: {best_xgb_rmse:.4f}")


In [ ]:
# Evaluasi model XGBoost terbaik pada Test Set
print("--- Evaluasi XGBoost Terbaik pada Data Testing ---")
y_pred_xgb_best_test = best_xgb_model.predict(X_test)
xgb_tuned_test_result = evaluate_model("XGB Tuned (Testing)", y_test, y_pred_xgb_best_test)

print(f"\nPerbandingan XGB Default vs XGB Tuned (pada Test Set):")
print(f"  Default --> RMSE: {xgb_default_test_result['rmse']:.4f} | MAE: {xgb_default_test_result['mae']:.4f} | R2: {xgb_default_test_result['r2']:.4f}")
print(f"  Tuned   --> RMSE: {xgb_tuned_test_result['rmse']:.4f} | MAE: {xgb_tuned_test_result['mae']:.4f} | R2: {xgb_tuned_test_result['r2']:.4f}")


## 6. Ringkasan Perbandingan Baseline


In [ ]:
# Tabel ringkasan semua model baseline
summary_data = [
    ['RF Default',  rf_default_test_result['rmse'],  rf_default_test_result['mae'],  rf_default_test_result['r2']],
    ['RF Tuned',    rf_tuned_test_result['rmse'],    rf_tuned_test_result['mae'],    rf_tuned_test_result['r2']],
    ['XGB Default', xgb_default_test_result['rmse'], xgb_default_test_result['mae'], xgb_default_test_result['r2']],
    ['XGB Tuned',   xgb_tuned_test_result['rmse'],   xgb_tuned_test_result['mae'],   xgb_tuned_test_result['r2']],
]

summary_df = pd.DataFrame(summary_data, columns=['Model', 'RMSE (ug/m3)', 'MAE (ug/m3)', 'R2'])
summary_df = summary_df.sort_values('RMSE (ug/m3)')

print("=" * 65)
print("RINGKASAN PERBANDINGAN MODEL BASELINE (Evaluasi pada Test Set)")
print("=" * 65)
print(summary_df.to_string(index=False))
print("=" * 65)
print(f"\nTarget LSTM/BiLSTM: Mengalahkan RMSE terbaik baseline di atas.")


## 7. Visualisasi Perbandingan (Time-Series)

Membandingkan prediksi model baseline terbaik (RF Tuned dan XGB Tuned) terhadap nilai aktual pada 2 minggu pertama data Test.


In [ ]:
# Ambil 14 hari pertama (14 * 24 jam = 336 baris)
subset_len = min(336, len(y_test))

time_axis = y_test.index[:subset_len]
actual    = rf_tuned_test_result['y_true'][:subset_len]
rf_pred   = rf_tuned_test_result['y_pred'][:subset_len]
xgb_pred  = xgb_tuned_test_result['y_pred'][:subset_len]

plt.figure(figsize=(16, 6))
plt.plot(time_axis, actual,   label='Aktual (PM2.5)', color='black', linewidth=2)
plt.plot(time_axis, rf_pred,  label='RF Tuned', color='blue', alpha=0.7, linestyle='--')
plt.plot(time_axis, xgb_pred, label='XGB Tuned', color='red', alpha=0.7, linestyle='-.')

plt.title("Prediksi Baseline vs Aktual (2 Minggu Pertama Data Test)", fontweight='bold')
plt.ylabel("PM2.5 (ug/m3)")
plt.xlabel("Waktu")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Menyimpan Model Terbaik

Model yang telah dioptimasi disimpan untuk digunakan ulang pada Fase 5 (SHAP Interpretation).


In [ ]:
MODEL_DIR = '/content/drive/MyDrive/KP_BRIN/models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Simpan Random Forest terbaik
joblib.dump(best_rf_model, os.path.join(MODEL_DIR, 'rf_tuned.pkl'))

# Simpan XGBoost terbaik
best_xgb_model.save_model(os.path.join(MODEL_DIR, 'xgb_tuned.json'))

# Simpan juga ringkasan hasil tuning
pd.DataFrame(rf_results).to_csv(os.path.join(MODEL_DIR, 'rf_tuning_results.csv'), index=False)
pd.DataFrame(xgb_results).to_csv(os.path.join(MODEL_DIR, 'xgb_tuning_results.csv'), index=False)

print(f"[OK] Model dan hasil tuning berhasil disimpan di: {MODEL_DIR}")
print(f"  -> rf_tuned.pkl             (Best RF: {best_rf_params})")
print(f"  -> xgb_tuned.json           (Best XGB: {best_xgb_params})")
print(f"  -> rf_tuning_results.csv    (Log semua kombinasi RF)")
print(f"  -> xgb_tuning_results.csv   (Log semua kombinasi XGB)")
